### Enzo Seiji Delgado Tabuchi - 573156

# RAG

In [4]:
%pip install --quiet --upgrade langchain-text-splitters langchain-community langgraph langchain-openai langchain-core pypdf unstructured

In [5]:
# configurando chatgpt
import getpass
import os
from google.colab import userdata
from langchain.chat_models import init_chat_model
from langchain_core.vectorstores import InMemoryVectorStore

In [6]:
os.environ["OPENAI_API_KEY"] = userdata.get('OPENAI_API_KEY')
llm = init_chat_model("gpt-4o-mini", model_provider="openai")

In [7]:
# selecionando o embedding
from langchain_openai import OpenAIEmbeddings

embeddings = OpenAIEmbeddings(model="text-embedding-ada-002")

In [9]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [10]:
vector_store = InMemoryVectorStore(embeddings)

In [12]:
# criando uma base de cohecimento
import bs4
from langchain_community.document_loaders import PyPDFLoader, WebBaseLoader

file_path = "./Manual_Candidato_FIAP_2026.pdf"
loader1 = PyPDFLoader(file_path)

loader2 = WebBaseLoader(
    ["https://www.ibm.com/br-pt/think/topics/generative-ai" ]
)

docs1 = loader1.load()
docs2 = loader2.load()
docs = docs1 + docs2

In [14]:
docs[0]

Document(metadata={'producer': 'ReportLab PDF Library - www.reportlab.com', 'creator': '(unspecified)', 'creationdate': '2026-05-16T00:21:50+00:00', 'author': '(anonymous)', 'keywords': '', 'moddate': '2026-05-16T00:21:50+00:00', 'subject': '(unspecified)', 'title': '(anonymous)', 'trapped': '/False', 'source': './Manual_Candidato_FIAP_2026.pdf', 'total_pages': 4, 'page': 0, 'page_label': '1'}, page_content='Manual do Candidato – FIAP 2026\n (Resumo Estruturado)\nEste documento resume as principais informações do vestibular da FIAP para ingresso em 2026,\nincluindo calendário, estrutura da avaliação, modalidades de graduação, matrículas, benefícios e\nrecomendações práticas para candidatos.\n1. Visão Geral da FIAP\nA FIAP é uma instituição brasileira de ensino superior focada em tecnologia, inovação, negócios digitais,\ninteligência artificial, segurança cibernética, engenharia, desenvolvimento de software e economia\ncriativa.\nOs formatos de ensino incluem:\n\x7f Graduação presencial

In [13]:
# fazendo o splitting dos documentos
from langchain_text_splitters import RecursiveCharacterTextSplitter

text_splitter = RecursiveCharacterTextSplitter(
    chunk_size=1000,  # chunk size (characters)
    chunk_overlap=200,  # chunk overlap (characters)
    add_start_index=True,  # track index in original document
)
all_splits = text_splitter.split_documents(docs)

print(f"Split pdf into {len(all_splits)} sub-documents.")

Split pdf into 61 sub-documents.


In [15]:
all_splits[0]

Document(metadata={'producer': 'ReportLab PDF Library - www.reportlab.com', 'creator': '(unspecified)', 'creationdate': '2026-05-16T00:21:50+00:00', 'author': '(anonymous)', 'keywords': '', 'moddate': '2026-05-16T00:21:50+00:00', 'subject': '(unspecified)', 'title': '(anonymous)', 'trapped': '/False', 'source': './Manual_Candidato_FIAP_2026.pdf', 'total_pages': 4, 'page': 0, 'page_label': '1', 'start_index': 0}, page_content='Manual do Candidato – FIAP 2026\n (Resumo Estruturado)\nEste documento resume as principais informações do vestibular da FIAP para ingresso em 2026,\nincluindo calendário, estrutura da avaliação, modalidades de graduação, matrículas, benefícios e\nrecomendações práticas para candidatos.\n1. Visão Geral da FIAP\nA FIAP é uma instituição brasileira de ensino superior focada em tecnologia, inovação, negócios digitais,\ninteligência artificial, segurança cibernética, engenharia, desenvolvimento de software e economia\ncriativa.\nOs formatos de ensino incluem:\n\x7f Gr

In [16]:
# guardando os dados em um banco de dados
document_ids = vector_store.add_documents(documents=all_splits)

print(document_ids[:3])

['5d876779-f63d-47ac-854b-889eedb73dc1', '9ee417b9-6bb8-4e7d-b1ae-993fa5e1489e', 'c35978fe-8305-4386-be4d-b013d207b8e7']


In [17]:
# Create a LANGSMITH_API_KEY in Settings > API Keys
from langsmith import Client
client = Client(api_key=userdata.get('LANGSMITH_API_KEY'))
prompt = client.pull_prompt("rlm/rag-prompt", include_model=True)

In [18]:
prompt

ChatPromptTemplate(input_variables=['context', 'question'], input_types={}, partial_variables={}, metadata={'lc_hub_owner': 'rlm', 'lc_hub_repo': 'rag-prompt', 'lc_hub_commit_hash': '50442af133e61576e74536c6556cefe1fac147cad032f4377b60c436e6cdcb6e'}, messages=[HumanMessagePromptTemplate(prompt=PromptTemplate(input_variables=['context', 'question'], input_types={}, partial_variables={}, template="You are an assistant for question-answering tasks. Use the following pieces of retrieved context to answer the question. If you don't know the answer, just say that you don't know. Use three sentences maximum and keep the answer concise.\nQuestion: {question} \nContext: {context} \nAnswer:"), additional_kwargs={})])

In [ ]:
# Como atualizar o prompt:

prompt.messages[0].prompt.template = '''You are an assistant for question-answering tasks. Use the following pieces of retrieved context to answer the question. If you don't know the answer, just say 'NAO SEI'. Use three sentences maximum and keep the answer concise.
Question: {question}
Context: {context}
Answer:'''

In [ ]:
from langchain_core.documents import Document
from typing_extensions import List, TypedDict


class State(TypedDict):
    question: str
    context: List[Document]
    answer: str

def retrieve(state: State):
    retrieved_docs = vector_store.similarity_search(state["question"])
    return {"context": retrieved_docs}


def generate(state: State):
    docs_content = "\n\n".join(doc.page_content for doc in state["context"])
    messages = prompt.invoke({"question": state["question"], "context": docs_content})
    response = llm.invoke(messages)
    return {"answer": response.content}

# definindo o workflow
from langgraph.graph import START, StateGraph

graph_builder = StateGraph(State).add_sequence([retrieve, generate])
graph_builder.add_edge(START, "retrieve")
graph = graph_builder.compile()

In [ ]:
result = graph.invoke({"question": "o que é machine learning?"})

print(f'Context: {result["context"]}\n\n')
print(f'Answer: {result["answer"]}')

In [ ]:

result = graph.invoke({"question": """9. Um dos riscos clássicos em sistemas RAG é:
A) Overfitting do classificador linear durante a etapa de inferência.
B) Recuperação de contexto irrelevante ou ambíguo, degradando a resposta final.
C) Impossibilidade de uso com documentos textuais longos.
D) Necessidade obrigatória de redes convolucionais.
E) Incompatibilidade com embeddings vetoriais."""})

In [ ]:
print(f'Answer: {result["answer"]}')

In [ ]:
result

# Exercícios

1 - Reproduzir o RAG do CP2 alimentado com o Manual do Candidato da FIAP.

2 - Fazer uma pesquisa sobre prompt injection e formas de quebrar um sistema RAG através do prompt do usuário.


3 - Desenvolver e comprovar por meio de testes 3 prompts capazes de fazer o seu RAG responder de maneira inadequada. Ex.: responder informação que não está contida na base de conhecimento, alucinar, devolver uma resposta errada.

